# Alpamayo 1 Demo

This notebook will load some example data from the NVIDIA [PhysicalAI-AV Dataset](https://huggingface.co/datasets/nvidia/PhysicalAI-Autonomous-Vehicles) and run the Alpamayo 1 model on it, producing and visualizing output trajectories and associated reasoning traces.

In [ ]:
import copy
import numpy as np
import pandas as pd

import torch
from alpamayo_r1.models.alpamayo_r1 import AlpamayoR1
from alpamayo_r1.load_physical_aiavdataset import load_physical_aiavdataset
from alpamayo_r1 import helper

### Load model and construct data preprocessor

In [ ]:
# check the gpu
!nvidia-smi

In [ ]:
model = AlpamayoR1.from_pretrained("nvidia/Alpamayo-R1-10B", dtype=torch.bfloat16).to("cuda")
processor = helper.get_processor(model.tokenizer)

### Load and prepare data

In [ ]:
clip_ids = pd.read_parquet("clip_ids.parquet")["clip_id"].tolist()
clip_id = clip_ids[774]
# examples
# 774 clip_id = '030c760c-ae38-49aa-9ad8-f5650a545d26'

data = load_physical_aiavdataset(clip_id)

messages = helper.create_message(data["image_frames"].flatten(0, 1))

inputs = processor.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=False,
    continue_final_message=True,
    return_dict=True,
    return_tensors="pt",
)
print("seq length:", inputs.input_ids.shape)
model_inputs = {
    "tokenized_data": inputs,
    "ego_history_xyz": data["ego_history_xyz"],
    "ego_history_rot": data["ego_history_rot"],
}
model_inputs = helper.to_device(model_inputs, "cuda")

### Inspect input data

In [ ]:
print("model_inputs:")
# full input data
for key, value in model_inputs.items():
    if isinstance(value, torch.Tensor):
        print(key, value.shape)
    else:
        print(key, type(value))

# data fed into model.vlm
print("\nmodel_inputs['tokenized_data']:")
for key, value in model_inputs["tokenized_data"].items():
    if isinstance(value, torch.Tensor):
        print(key, value.shape)
    else:
        print(key, type(value))


### Visualization of the ego history in BEV

In [ ]:
import matplotlib.pyplot as plt

# Extract ego history — (1, 1, T, 3) → (T, 3)
ego_xyz = model_inputs["ego_history_xyz"].cpu().numpy().squeeze()  # (T, 3)

x, y = ego_xyz[:, 0], ego_xyz[:, 1]  # ignore z
T = len(x)

# BEV mapping: plot_x = -y (right), plot_y = x (forward up)
px, py = -y, x

fig, ax = plt.subplots(figsize=(8, 8))

# Trajectory line + markers
ax.plot(px, py, color="royalblue", linewidth=2, alpha=0.6, zorder=2)
ax.scatter(px, py, c=np.arange(T), cmap="viridis", s=30, zorder=3)

# Start & End
ax.scatter(px[0], py[0], color="green", s=100, marker="D", zorder=4, label="Ego History Start")
ax.scatter(px[-1], py[-1], color="red", s=100, marker="D", zorder=4, label="Ego History End (t0)")

ax.set_xlabel("Y (right) [m]")
ax.set_ylabel("X (forward) [m]")
ax.set_title("BEV Ego History")
ax.set_aspect("equal")

# Make plot square
v_span = py.max() - py.min()
h_span = px.max() - px.min()
span = max(v_span, h_span) * 1.1  # 10% padding
h_mid = (px.max() + px.min()) / 2
v_mid = (py.max() + py.min()) / 2
ax.set_xlim(h_mid - span / 2, h_mid + span / 2)
ax.set_ylim(v_mid - span / 2, v_mid + span / 2)

ax.legend()

ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt

# Camera names corresponding to the sorted camera indices
# Map camera indices to names (based on avdi.features.CAMERA enum)
CAMERA_INDEX_TO_NAME = {
    0: "Cross Left",
    1: "Front Wide",
    2: "Cross Right",
    6: "Front Tele",
}
frame_idx_to_time = [
    f"t={t - data['relative_timestamps'][-1].tolist()[-1]:.1f}s"
    for t in data["relative_timestamps"][-1].tolist()
]
camera_names = [
    CAMERA_INDEX_TO_NAME.get(idx.item(), f"Camera {idx.item()}") for idx in data["camera_indices"]
]

# image_frames shape: (N_cameras, num_frames, 3, H, W)
images = (
    data["image_frames"].permute(0, 1, 3, 4, 2).cpu().numpy()
)  # (N_cameras, num_frames, H, W, 3)
n_cameras, n_frames = images.shape[:2]

fig, axes = plt.subplots(n_cameras, n_frames, figsize=(12, 8))
plt.subplots_adjust(wspace=0.02, hspace=0.1, left=0.12, right=0.98, top=0.92, bottom=0.02)

for cam_idx in range(n_cameras):
    for frame_idx in range(n_frames):
        ax = axes[cam_idx, frame_idx]
        ax.imshow(images[cam_idx, frame_idx])
        ax.axis("off")

        # Add column labels (timesteps) at the top row
        if cam_idx == 0:
            ax.set_title(frame_idx_to_time[frame_idx], fontsize=11)

        # Add row labels (camera names) on the left column
        if frame_idx == 0:
            ax.text(
                -0.05,
                0.5,
                camera_names[cam_idx],
                fontsize=10,
                rotation=0,
                ha="right",
                va="center",
                transform=ax.transAxes,
            )

plt.show()

In [ ]:
# check the input sequence
input_ids = model_inputs["tokenized_data"]["input_ids"]
tokenizer = model.tokenizer
input_seq = tokenizer.decode(input_ids[0])

# replacing special tokens for easier reading
input_seq = input_seq.replace("<|image_pad|>", "I")
input_seq = input_seq.replace("<|traj_history|>", "H")
input_seq = input_seq.replace("<|traj_future|>", "F")
input_seq = input_seq.replace("<|vision_end|>", "<|vision_end|>\n")
input_seq = input_seq.replace("<|traj_history_end|>", "<|traj_history_end|>\n")

print(input_seq)


### Model inference

In [ ]:
torch.cuda.manual_seed_all(42)
with torch.autocast("cuda", dtype=torch.bfloat16):
    pred_xyz, pred_rot, extra = model.sample_trajectories_from_data_with_vlm_rollout(
        data=copy.deepcopy(model_inputs),
        top_p=0.98,
        temperature=0.6,
        num_traj_samples=4,  # Feel free to raise this for more output trajectories and CoC traces.
        max_generation_length=256,
        return_extra=True,
    )

# the size is [batch_size, num_traj_sets, num_traj_samples]
print("Chain-of-Causation (per trajectory):\n", extra["cot"][0])

## Visualizing data and results

In [ ]:
import matplotlib.pyplot as plt

# --- Ego history (BEV) ---
hist_xyz = model_inputs["ego_history_xyz"].cpu().numpy().squeeze()  # (T_h, 3)
hx, hy = -hist_xyz[:, 1], hist_xyz[:, 0]  # BEV: plot_x=-y, plot_y=x

# --- GT future ---
gt_xyz = data["ego_future_xyz"].cpu().numpy().squeeze()  # (T_f, 3)
gx, gy = -gt_xyz[:, 1], gt_xyz[:, 0]

# --- Predicted futures ---
n_samples = pred_xyz.shape[2]
pred_np = pred_xyz.cpu().numpy()[0, 0]  # (n_samples, T_f, 3)

fig, ax = plt.subplots(figsize=(8, 8))

# Ego history
ax.plot(hx, hy, color="royalblue", linewidth=2, alpha=0.6, zorder=2)
ax.scatter(hx, hy, c=np.arange(len(hx)), cmap="viridis", s=30, zorder=3)
ax.scatter(hx[0], hy[0], color="green", s=100, marker="D", zorder=4, label="History Start")
ax.scatter(hx[-1], hy[-1], color="red", s=100, marker="D", zorder=4, label="t0")

# Predicted trajectories
colors = plt.cm.tab10(np.linspace(0, 1, n_samples))
for i in range(n_samples):
    px_pred = -pred_np[i, :, 1]
    py_pred = pred_np[i, :, 0]
    ax.plot(
        px_pred,
        py_pred,
        "o-",
        color=colors[i],
        markersize=4,
        linewidth=1.5,
        alpha=0.8,
        zorder=5,
        label=f"Pred #{i + 1}",
    )

# GT future
ax.plot(gx, gy, "s-", color="red", markersize=4, linewidth=2, zorder=6, label="GT Future")

ax.set_xlabel("Y (right) [m]")
ax.set_ylabel("X (forward) [m]")
ax.set_title("BEV: Ego History + Predicted & GT Trajectories")
ax.set_aspect("equal")

# Make plot square
all_px = np.concatenate([hx, gx] + [-pred_np[i, :, 1] for i in range(n_samples)])  # horizontal
all_py = np.concatenate([hy, gy] + [pred_np[i, :, 0] for i in range(n_samples)])  # vertical
v_span = all_py.max() - all_py.min()
h_span = all_px.max() - all_px.min()
span = max(v_span, h_span) * 1.1  # 10% padding
h_mid = (all_px.max() + all_px.min()) / 2
v_mid = (all_py.max() + all_py.min()) / 2
ax.set_xlim(h_mid - span / 2, h_mid + span / 2)
ax.set_ylim(v_mid - span / 2, v_mid + span / 2)

ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
pred_xy = pred_xyz.cpu().numpy()[0, 0, :, :, :2].transpose(0, 2, 1)
gt_xy = data["ego_future_xyz"].cpu().numpy()[0, 0, :, :2].T
diff = np.linalg.norm(pred_xy - gt_xy[None, ...], axis=1).mean(-1)
print("minADE:", diff.min(), "meters")